In [17]:
%pip install langchain openai beautifulsoup4 playwright faiss-cpu tiktoken python-dotenv nest_asyncio playwright langchain-community
import subprocess
subprocess.run(["playwright", "install"])

Note: you may need to restart the kernel to use updated packages.


CompletedProcess(args=['playwright', 'install'], returncode=0)

In [7]:
from bs4 import BeautifulSoup
import nest_asyncio
import asyncio
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain.chat_models import ChatOpenAI
from langchain.chains import RetrievalQA
from dotenv import load_dotenv
import os
from urllib.parse import urljoin, urlparse
from playwright.async_api import async_playwright


In [8]:
BASE_URL="https://victorsna.github.io"
START_PATH = "/wiki_ai/#/"

In [9]:
# Carrega variáveis de ambiente do .env
load_dotenv()

# Garante que a chave da OpenAI está disponível
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("A chave OPENAI_API_KEY não foi encontrada no arquivo .env")

In [10]:
nest_asyncio.apply()


In [51]:
async def crawl_docs(base_url, start_path, max_pages=10):
    visited = set()
    to_visit = [start_path]
    documents = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        while to_visit and len(visited) < max_pages:
            path = to_visit.pop(0)
            if path in visited:
                continue

            url = urljoin(base_url, path)
            print(f"Crawling: {url}")
            try:
                await page.goto(url, timeout=60000)
                await page.wait_for_timeout(2000)
                text = await page.inner_text("body")

                documents.append(Document(page_content=text, metadata={"source": url}))

                # Coleta links internos da página já renderizada
                hrefs = await page.eval_on_selector_all("a", "elements => elements.map(el => el.href)")
                for href in hrefs:
                    parsed = urlparse(href)
                    
                    if href not in visited and href not in to_visit and parsed.query == "":
                        rel_path = parsed.path
                        fragment = parsed.fragment

                        to_visit.append(rel_path[:-1] + "#" + fragment)
            except Exception as e:
                print(f"Erro ao acessar {url}: {e}")

            visited.add(path)

        await browser.close()

    return documents

In [52]:
raw_docs = await crawl_docs(BASE_URL, START_PATH, max_pages=15)

Crawling: https://victorsna.github.io/wiki_ai/#/
Crawling: https://victorsna.github.io/wiki_ai#/
Crawling: https://victorsna.github.io/wiki_ai#/cart/README
Crawling: https://victorsna.github.io/wiki_ai#/?id=supermecado-app
Crawling: https://victorsna.github.io/wiki_ai#/?id=description
Crawling: https://victorsna.github.io/wiki_ai#/cart/README?id=post-apiv1cartadd
Crawling: https://victorsna.github.io/wiki_ai#/cart/README?id=request-headers
Crawling: https://victorsna.github.io/wiki_ai#/cart/README?id=request-body
Crawling: https://victorsna.github.io/wiki_ai#/cart/README?id=response-200-ok
Crawling: https://victorsna.github.io/wiki_ai#/cart/README?id=response-400-bad-request
Crawling: https://victorsna.github.io/wiki_ai#/cart/README?id=response-401-unauthorized
Crawling: https://victorsna.github.io/wiki_ai#/cart/README?id=notes


In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
    docs = splitter.split_documents(raw_docs)

    embeddings = OpenAIEmbeddings()
    vectorstore = FAISS.from_documents(docs, embeddings)

    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
    qa_chain = RetrievalQA.from_chain_type(
        llm=ChatOpenAI(model_name="gpt-3.5-turbo"),
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=True
    )

    query = "How to deploy a Streamlit app to the cloud?"
    result = qa_chain(query)

    print("\nResposta:\n", result["result"])
    print("\nFontes:\n")
    for doc in result["source_documents"]:
        print(doc.metadata["source"])

[Document(metadata={'source': 'https://victorsna.github.io/wiki_ai/#/'}, page_content=' Home\n\n🛒 Cart\n\nSupermecado App\nDescription\n\nSupermercado+ is your smart shopping companion, designed to make grocery shopping faster, easier, and more convenient. Whether you’re planning your weekly meals, stocking up on household essentials, or just grabbing a few items, Supermercado+ has everything you need—right at your fingertips.\n\nWith a clean and intuitive interface, the app allows you to browse a wide selection of fresh produce, pantry staples, frozen goods, beverages, cleaning supplies, personal care items, and more. Each product comes with detailed descriptions, nutritional information, prices, and customer reviews, helping you make informed choices.\n\nUse the built-in smart search and filters to quickly find exactly what you’re looking for. Create custom shopping lists, track your spending in real time, and even receive personalized recommendations based on your previous purchases